In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
data = load_breast_cancer()
X, y = data.data, data.target
target_names = data.target_names

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
model_configs = {
    "K-Nearest Neighbors (KNN)": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier())
        ]),
        "params": {
            "clf__n_neighbors": [3, 5, 7, 9, 11],
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "manhattan"]
        }
    },
    "Gaussian Naive Bayes": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", GaussianNB())
        ]),
        "params": {
            "clf__var_smoothing": np.logspace(-11, -7, 5)
        }
    },
    "Support Vector Machine (SVM - RBF)": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(probability=True, random_state=42))
        ]),
        "params": {
            "clf__C": [0.1, 1, 10, 100],
            "clf__gamma": ["scale", "auto", 0.01, 0.1]
        }
    }
}

In [6]:
line_break = "-" * 80

In [7]:
for name, config in model_configs.items():

    grid_search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    print(f"\n{line_break}")
    print(f" MODEL: {name}")
    print(f" Best Params: {grid_search.best_params_}")
    print(line_break)
    
  
    print(classification_report(y_test, y_pred, target_names=target_names, digits=4))
    
    cm = confusion_matrix(y_test, y_pred)
    print(f"Confusion Matrix [Rows: Actual, Cols: Predicted]:")
    print(f"  Malignant (0):  {cm[0][0]:>3} TN  | {cm[0][1]:>3} FP")
    print(f"  Benign    (1):  {cm[1][0]:>3} FN  | {cm[1][1]:>3} TP")


--------------------------------------------------------------------------------
 MODEL: K-Nearest Neighbors (KNN)
 Best Params: {'clf__metric': 'manhattan', 'clf__n_neighbors': 3, 'clf__weights': 'uniform'}
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

   malignant     0.9750    0.9286    0.9512        42
      benign     0.9595    0.9861    0.9726        72

    accuracy                         0.9649       114
   macro avg     0.9672    0.9573    0.9619       114
weighted avg     0.9652    0.9649    0.9647       114

Confusion Matrix [Rows: Actual, Cols: Predicted]:
  Malignant (0):   39 TN  |   3 FP
  Benign    (1):    1 FN  |  71 TP

--------------------------------------------------------------------------------
 MODEL: Gaussian Naive Bayes
 Best Params: {'clf__var_smoothing': np.float64(1e-11)}
--------------------------------------------------------------------------------
              p

c:\Users\offsh\Documents\project\tf_env\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
